In [2]:
!pip3 install kagglehub --upgrade --user --quiet


[notice] A new release of pip is available: 25.1.1 -> 26.1.1
[notice] To update, run: pip install --upgrade pip


In [4]:
# Import dataset

# Install dependencies as needed:
# pip install kagglehub[pandas-datasets]
import kagglehub
from kagglehub import KaggleDatasetAdapter

file_path = "train.csv"

# Load the latest version
df = kagglehub.load_dataset(
  KaggleDatasetAdapter.PANDAS,
  "prakharrathi25/banking-dataset-marketing-targets",
  file_path,
)

print("First 5 records:", df.head())

/var/folders/ls/1wdsysgs1v18dvs6sj0dbplh0000gn/T/ipykernel_41592/2762164146.py:11: DeprecationWarning: Use dataset_load() instead of load_dataset(). load_dataset() will be removed in a future version.
  df = kagglehub.load_dataset(


100%|██████████| 512k/512k [00:00<00:00, 1.22MB/s]

Extracting zip of train.csv...
First 5 records:   age;"job";"marital";"education";"default";"balance";"housing";"loan";"contact";"day";"month";"duration";"campaign";"pdays";"previous";"poutcome";"y"
0  58;"management";"married";"tertiary";"no";2143...                                                                                                  
1  44;"technician";"single";"secondary";"no";29;"...                                                                                                  
2  33;"entrepreneur";"married";"secondary";"no";2...                                                                                                  
3  47;"blue-collar";"married";"unknown";"no";1506...                                                                                                  
4  33;"unknown";"single";"unknown";"no";1;"no";"n...                                                                                                  


In [ ]:
# Import statements
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.neighbors import KNeighborsClassifier
import numpy as np
from sklearn.model_selection import train_test_split


In [ ]:
# Visualise


In [ ]:
# Preprocessing structure through pipeline
def pipeline(filename):
  # Missing Removal
  rows_not_missing = df.total_bedrooms.notna()
  rows_missing = df.total_bedrooms.isna()
  df_without_missings = df[rows_not_missing]
  knn = KNeighborsClassifier(n_neighbors=1)
  knn.fit(df_without_missings, df.total_bedrooms[rows_not_missing])
  imputed_values = knn.predict(df[rows_missing]);
  df.loc[rows_missing, 'total_bedrooms'] = imputed_values

  # Ourlier Removal
  numeric_df = df.select_dtypes(include=np.number)
  Q1 = numeric_df.quantile(0.25, numeric_only=True)
  Q3 = numeric_df.quantile(0.75, numeric_only=True)
  IQR = Q3 - Q1
  for col in numeric_df.columns:
      lower_bound = Q1[col] - 1.5 * IQR[col]
      upper_bound = Q3[col] + 1.5 * IQR[col]
      df[col] = np.clip(df[col], lower_bound, upper_bound)

  # Adding features
  housing['rooms_per_household'] = housing['total_rooms'] / housing['households']
  housing['bedrooms_per_room'] = housing['total_bedrooms'] / housing['total_rooms']
  housing['population_per_household'] = housing['population'] / housing['households']

  # Categorical Encoding
  housing_cat = housing[['ocean_proximity']]
  housing_dummies = pd.get_dummies(housing_cat, prefix='ocean_proximity', dtype=int)
  housing = pd.concat([housing.drop('ocean_proximity', axis=1), housing_dummies], axis=1)

  # Numerical Transformation
  std_scaler = preprocessing.StandardScaler()
  housing[housing.columns] = std_scaler.fit_transform(housing[housing.columns])

  return housing

df = pipeline("housing.csv")
df.head()
